# PyTorch torch.nn.Linear Relationships Analysis

This notebook demonstrates how to analyze the relationships of `torch.nn.Linear` using the comprehensive knowledge graph.

In [1]:
# Import required libraries
import json
import networkx as nx
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

print("Loading knowledge graph...")

# Load the comprehensive knowledge graph
with open('comprehensive_knowledge_graph.json', 'r') as f:
    knowledge_graph = json.load(f)

print(f"Graph loaded with {len(knowledge_graph['nodes'])} nodes and {len(knowledge_graph['edges'])} edges")

Loading knowledge graph...
Graph loaded with 143311 nodes and 25716244 edges


## Build NetworkX Graph

Convert the knowledge graph to NetworkX format for easy analysis and visualization.

In [2]:
def build_nx_graph(knowledge_graph):
    """Build NetworkX graph from knowledge graph data."""
    G = nx.DiGraph()

    # Add nodes
    for node in knowledge_graph['nodes']:
        G.add_node(
            node['id'],
            name=node['name'],
            type=node['type'],
            path=node.get('path', ''),
            description=node.get('description', ''),
            module=node.get('module', '')
        )

    # Add edges
    edges_data = knowledge_graph.get('edges', [])
    for link in edges_data:
        G.add_edge(
            link['source'],
            link['target'],
            link_type=link['type'],
            **link.get('properties', {})
        )

    return G

# Build the NetworkX graph
G = build_nx_graph(knowledge_graph)
print(f"NetworkX graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

NetworkX graph built with 97272 nodes and 10369336 edges


## Find torch.nn.Linear

Locate the torch.nn.Linear class in the knowledge graph.

In [3]:
# Find torch.nn.Linear
linear_node_id = None
for node_id, node_data in G.nodes(data=True):
    if node_data.get('name') == 'Linear' and 'torch/nn/modules/linear.py' in node_data.get('path', ''):
        linear_node_id = node_id
        break

if not linear_node_id:
    print("Could not find torch.nn.Linear in the knowledge graph")
    # Try to find any Linear class
    linear_nodes = [node_id for node_id, node_data in G.nodes(data=True)
                   if node_data.get('name') == 'Linear']
    if linear_nodes:
        print("Found Linear classes:")
        for node_id in linear_nodes[:5]:
            print(f"  {G.nodes[node_id].get('name')} in {G.nodes[node_id].get('path')}")
    else:
        print("No Linear classes found")
else:
    print(f"Found torch.nn.Linear with node ID: {linear_node_id}")
    linear_node_data = G.nodes[linear_node_id]
    print(f"  Name: {linear_node_data.get('name')}")
    print(f"  Type: {linear_node_data.get('type')}")
    print(f"  Path: {linear_node_data.get('path')}")
    print(f"  Module: {linear_node_data.get('module')}\n")

Found torch.nn.Linear with node ID: torch/nn/modules/linear.py:Linear
  Name: Linear
  Type: class
  Path: torch/nn/modules/linear.py
  Module: nn.modules.linear



## Analyze Linear Relationships

Examine the relationships of torch.nn.Linear with its neighbors.

In [4]:
if linear_node_id:
    # Get neighbors
    predecessors = list(G.predecessors(linear_node_id))
    successors = list(G.successors(linear_node_id))
    
    print(f"torch.nn.Linear has {len(predecessors)} incoming relationships and {len(successors)} outgoing relationships")
    
    # Show incoming relationships (what uses Linear)
    print("\nIncoming relationships (what uses Linear):")
    for i, pred in enumerate(predecessors[:10]):
        pred_data = G.nodes[pred]
        print(f"  {i+1}. {pred_data.get('name', pred)} ({pred_data.get('type', 'unknown')}) in {pred_data.get('path', 'no path')}")
        
    # Show outgoing relationships (what Linear uses)
    print("\nOutgoing relationships (what Linear uses):")
    for i, succ in enumerate(successors[:10]):
        succ_data = G.nodes[succ]
        edge_data = G.get_edge_data(linear_node_id, succ)
        edge_type = edge_data.get('link_type', 'unknown') if edge_data else 'unknown'
        print(f"  {i+1}. {succ_data.get('name', succ)} ({succ_data.get('type', 'unknown')}) [{edge_type}]")
        
    # Show inheritance relationships specifically
    inheritance_edges = [(u, v, data) for u, v, data in G.edges(data=True)
                        if data.get('link_type') == 'inheritance' and u == linear_node_id]
    
    print("\nInheritance relationships:")
    for u, v, data in inheritance_edges:
        v_data = G.nodes[v]
        print(f"  {v_data.get('name', v)} inherits from Linear")

torch.nn.Linear has 3 incoming relationships and 13 outgoing relationships

Incoming relationships (what uses Linear):
  1. linear.py (file) in torch/nn/modules/linear.py
  2. __all__ (variable) in torch/nn/modules/linear.py
  3. Identity (class) in torch/nn/modules/linear.py

Outgoing relationships (what Linear uses):
  1. Module (class) [inheritance]
  2. NonDynamicallyQuantizableLinear (class) [module_internal]
  3. Bilinear (class) [module_internal]
  4. LazyLinear (class) [module_internal]
  5. __init__ (function) [module_internal]
  6. forward (function) [module_internal]
  7. __constants__ (variable) [module_internal]
  8. reset_parameters (function) [module_internal]
  9. extra_repr (function) [module_internal]
  10. cls_to_become (variable) [module_internal]

Inheritance relationships:
  Module inherits from Linear


## Visualize Linear's Neighbors

Create a visualization of torch.nn.Linear and its direct neighbors (fixed positioning).

In [5]:
if linear_node_id:
    # Create a subgraph with Linear and its neighbors
    neighbors = list(G.predecessors(linear_node_id)) + list(G.successors(linear_node_id))
    subgraph_nodes = [linear_node_id] + neighbors
    
    # Get edges involving these nodes
    subgraph_edges = []
    for u, v, data in G.edges(data=True):
        if u in subgraph_nodes and v in subgraph_nodes:
            subgraph_edges.append((u, v, data))
    
    # Create subgraph
    H = nx.DiGraph()
    for node_id in subgraph_nodes:
        H.add_node(node_id, **G.nodes[node_id])
    
    for u, v, data in subgraph_edges:
        H.add_edge(u, v, **data)
    
    print(f"Created subgraph with {H.number_of_nodes()} nodes and {H.number_of_edges()} edges")
    
    # Visualize the subgraph with fixed positioning
    plt.figure(figsize=(15, 12))
    
    # Position nodes using spring layout
    try:
        pos = nx.spring_layout(H, k=1, iterations=50)
    except Exception as e:
        print(f"Error creating layout: {e}")
        # Fallback to circular layout
        pos = nx.circular_layout(H)
    
    # Draw edges
    edge_colors = {'inheritance': 'purple', 'module_internal': 'blue', 'other': 'gray'}
    
    # Draw edges grouped by type
    for edge_type in set(d.get('link_type', 'other') for _, _, d in H.edges(data=True)):
        edges = [(u, v) for u, v, d in H.edges(data=True) if d.get('link_type') == edge_type]
        if edges:
            color = edge_colors.get(edge_type, edge_colors['other'])
            width = 2 if edge_type == 'inheritance' else 1
            alpha = 0.7
            nx.draw_networkx_edges(
                H, pos,
                edgelist=edges,
                edge_color=color,
                width=width,
                alpha=alpha,
                arrows=True,
                arrowsize=15,
                connectionstyle='arc3,rad=0.1'
            )
    
    # Draw nodes
    node_colors = []
    for node_id in H.nodes():
        node_type = H.nodes[node_id].get('type', 'other')
        if node_type == 'class':
            node_colors.append('lightblue')
        elif node_type == 'function':
            node_colors.append('lightgreen')
        elif node_type == 'variable':
            node_colors.append('lightyellow')
        else:
            node_colors.append('lightgray')
    
    nx.draw_networkx_nodes(H, pos, node_color=node_colors, alpha=0.8, node_size=1500)
    
    # Draw labels
    labels = {}
    for node_id in H.nodes():
        name = H.nodes[node_id].get('name', node_id)
        labels[node_id] = name
    
    nx.draw_networkx_labels(H, pos, labels, font_size=8)
    
    # Fix the positioning issue for annotation
    # Get the position of Linear node properly
    try:
        linear_pos = pos.get(linear_node_id, None)
        if linear_pos is not None and len(linear_pos) >= 2:
            # Make sure we're using proper array indexing
            x_pos = float(linear_pos[0]) if hasattr(linear_pos[0], '__float__') else 0
            y_pos = float(linear_pos[1]) if hasattr(linear_pos[1], '__float__') else 0
            
            plt.annotate('torch.nn.Linear',
                       xy=(x_pos, y_pos),
                       xytext=(10, 10),
                       textcoords='offset points',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.9),
                       fontsize=12, fontweight='bold')
        else:
            print("Could not get proper position for annotation")
    except Exception as e:
        print(f"Error with annotation positioning: {e}")
    
    plt.title('torch.nn.Linear Relationships', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    
    # Save the visualization
    plt.savefig('linear_relationships.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Visualization saved as 'linear_relationships.png'")

Created subgraph with 17 nodes and 143 edges
Visualization saved as 'linear_relationships.png'


## Detailed Analysis of Linear Components

Examine the detailed structure of torch.nn.Linear and its related components.

In [6]:
if linear_node_id:
    print("=== Detailed torch.nn.Linear Analysis ===")
    
    # Get Linear node data
    linear_data = G.nodes[linear_node_id]
    print(f"Name: {linear_data.get('name')}")
    print(f"Type: {linear_data.get('type')}")
    print(f"Path: {linear_data.get('path')}")
    print(f"Module: {linear_data.get('module')}")
    
    # Show all relationships
    print("\nAll Relationships:")
    
    # Incoming relationships
    incoming = list(G.predecessors(linear_node_id))
    print(f"Incoming relationships ({len(incoming)}):")
    for i, pred in enumerate(incoming):
        pred_data = G.nodes[pred]
        print(f"  {i+1}. {pred_data.get('name')} ({pred_data.get('type')})")
        
    # Outgoing relationships
    outgoing = list(G.successors(linear_node_id))
    print(f"\nOutgoing relationships ({len(outgoing)}):")
    for i, succ in enumerate(outgoing):
        succ_data = G.nodes[succ]
        edge_data = G.get_edge_data(linear_node_id, succ)
        edge_type = edge_data.get('link_type', 'unknown') if edge_data else 'unknown'
        print(f"  {i+1}. {succ_data.get('name')} ({succ_data.get('type')}) [{edge_type}]")

=== Detailed torch.nn.Linear Analysis ===
Name: Linear
Type: class
Path: torch/nn/modules/linear.py
Module: nn.modules.linear

All Relationships:
Incoming relationships (3):
  1. linear.py (file)
  2. __all__ (variable)
  3. Identity (class)

Outgoing relationships (13):
  1. Module (class) [inheritance]
  2. NonDynamicallyQuantizableLinear (class) [module_internal]
  3. Bilinear (class) [module_internal]
  4. LazyLinear (class) [module_internal]
  5. __init__ (function) [module_internal]
  6. forward (function) [module_internal]
  7. __constants__ (variable) [module_internal]
  8. reset_parameters (function) [module_internal]
  9. extra_repr (function) [module_internal]
  10. cls_to_become (variable) [module_internal]
  11. initialize_parameters (function) [module_internal]
  12. factory_kwargs (variable) [module_internal]
  13. bound (variable) [module_internal]


## Key Subsystems Related to Linear

Analyze which PyTorch subsystems are related to Linear.

In [7]:
print("=== PyTorch Subsystems Related to Linear ===")

# Find Linear-related components
linear_related = []
for node_id, node_data in G.nodes(data=True):
    if 'Linear' in node_data.get('name', '') and node_data.get('type') == 'class':
        # Skip the main Linear class
        if 'torch/nn/modules/linear.py:Linear' != node_id:
            linear_related.append(node_data)

print(f"Found {len(linear_related)} Linear-related classes:")
for i, node in enumerate(linear_related[:10]):
    print(f"  {i+1}. {node.get('name')} in {node.get('path')}")
    
if len(linear_related) > 10:
    print(f"  ... and {len(linear_related) - 10} more")
    
# Show quantized Linear variants
quantized_linear = [node for node in G.nodes(data=True) 
                   if 'quantized' in node[1].get('module', '') and 'Linear' in node[1].get('name', '')]
print(f"\nQuantized Linear variants: {len(quantized_linear)}")
for i, node in enumerate(quantized_linear[:5]):
    print(f"  {i+1}. {node[1].get('name')} in {node[1].get('path')}")

=== PyTorch Subsystems Related to Linear ===
Found 86 Linear-related classes:
  1. MKLPackedLinear in torch/_inductor/mkldnn_ir.py
  2. LinearUnary in torch/_inductor/mkldnn_ir.py
  3. LinearBinary in torch/_inductor/mkldnn_ir.py
  4. QLinearPointwisePT2E in torch/_inductor/mkldnn_ir.py
  5. QLinearPointwiseBinaryPT2E in torch/_inductor/mkldnn_ir.py
  6. PostGradBatchLinearFusion in torch/_inductor/fx_passes/group_batch_fusion.py
  7. GroupLinearFusion in torch/_inductor/fx_passes/group_batch_fusion.py
  8. BatchLinearLHSFusion in torch/_inductor/fx_passes/group_batch_fusion.py
  9. PreGradBatchLinearFusion in torch/_inductor/fx_passes/group_batch_fusion.py
  10. NormalizedLinearNode in torch/_inductor/fx_passes/pre_grad.py
  ... and 76 more

Quantized Linear variants: 14
  1. LinearReLU in torch/ao/nn/intrinsic/quantized/dynamic/modules/linear_relu.py
  2. LinearReLU in torch/ao/nn/intrinsic/quantized/modules/linear_relu.py
  3. LinearLeakyReLU in torch/ao/nn/intrinsic/quantized/modul

## Graph Statistics

Show overall statistics of the knowledge graph.

In [8]:
print("=== Knowledge Graph Statistics ===")
print(f"Total Nodes: {G.number_of_nodes()}")
print(f"Total Edges: {G.number_of_edges()}")

# Node type distribution
node_types = Counter(node_data.get('type', 'other') for _, node_data in G.nodes(data=True))
print("\nNode Type Distribution:")
for node_type, count in node_types.most_common():
    print(f"  {node_type}: {count}")

# Edge type distribution
edge_types = Counter(data.get('link_type', 'other') for _, _, data in G.edges(data=True))
print("\nEdge Type Distribution:")
for edge_type, count in edge_types.most_common():
    print(f"  {edge_type}: {count}")

=== Knowledge Graph Statistics ===
Total Nodes: 97272
Total Edges: 10369336

Node Type Distribution:
  variable: 55436
  function: 34493
  class: 5161
  file: 2182

Edge Type Distribution:
  module_internal: 10367125
  inheritance: 2211


## Conclusion

This notebook demonstrates how to analyze the relationships of torch.nn.Linear using the comprehensive knowledge graph. Key insights:

1. **Linear inherits from Module**: The core relationship showing Linear is a neural network module
2. **Module Internal Relationships**: Shows components within the same module that Linear interacts with
3. **Complete Architecture**: Provides full view of Linear's place in the PyTorch ecosystem
4. **Visualization**: Creates clear visual representation of Linear's neighbors
5. **Subsystems**: Shows related components like quantized variants and fused modules

The knowledge graph now enables comprehensive analysis of PyTorch's architecture and component relationships.

In [9]:
linear_node_id

'torch/nn/modules/linear.py:Linear'

In [10]:
linear_data

{'name': 'Linear',
 'type': 'class',
 'path': 'torch/nn/modules/linear.py',
 'description': '',
 'module': 'nn.modules.linear'}

In [12]:
G.predecessors(linear_node_id)